In [1]:
import pandas as pd
import scipy as sp
import numpy as np
from lightfm import cross_validation
from lightfm.evaluation import precision_at_k
from lightfm.evaluation import recall_at_k
from lightfm.evaluation import auc_score
from lightfm import LightFM
import matplotlib.pyplot as plt
import pickle

In [7]:
#load data
data = []
with open("/home/nasi4/proj/GWAS/UKBB_201807/results/all.UKBB.hg38.sig.phenotype.bed")as f:
    for line in f:
        data.append(line.strip().split())

In [8]:
#combine label elements
for row in data:
    row.append(' '.join(row[5:]))
    del row[5:-1]

In [9]:
#create dataframe
df = pd.DataFrame(data, columns = ['Chromosome', 'p1', 'p2', 'Reference SNP cluster ID', 'p-value', 'Trait'])

In [10]:
df = df[df['p-value']!='0.00000e+00']
test_list = df['p-value'].to_list()
test_list = list(map(float, test_list))
print(max(test_list))

5e-08


In [11]:
#calculating the negative log of the p-value, setting infinite values to the max non-infinite value observed, and
#normalizing by max -log(p-value) score
pval_mlog = -np.log(list(map(float, list(df['p-value']))))

inf_mask = np.ma.array(pval_mlog, mask=~np.isfinite(pval_mlog))
max_ind = np.ma.argmax(inf_mask, fill_value=-np.inf)
print(max_ind)
max_val = pval_mlog[max_ind]
print(max_val)

for ind in range(len(pval_mlog)):
    if pval_mlog[ind] > pval_mlog[max_ind]:
        pval_mlog[ind] = pval_mlog[max_ind]
        
df['pval_mlog'] = pval_mlog/pval_mlog[max_ind]

736764
743.7469247408213


In [12]:
#creating df with SNP, trait, and p-value
ts_df = df[['Reference SNP cluster ID', 'Trait', 'pval_mlog']]
ts_df.head()

,Reference SNP cluster ID,Trait,pval_mlog
0,rs547522712,Diagnoses - main ICD10: L98 Other disorders of...,0.061532
1,rs547522712,Other disorders of skin and subcutaneous tissu...,0.061532
2,rs547522712,#Spondylopathies,0.058265
3,rs547522712,Diagnoses - main ICD10: M48 Other spondylopathies,0.122894
4,rs547522712,Other and unspecified disorders of skin and su...,0.066955


In [13]:
#finding unique traits and SNPs
SNP_values = np.unique(ts_df['Reference SNP cluster ID'].values)
trait_values = np.unique(ts_df['Trait'].values)

In [14]:
print(trait_values[0:50])

['#Arthrosis' '#Deforming dorsopathies' '#Other joint disorders'
 '#Other necrotizing vasculopathies'
 '#Other necrotizing vasculopathies (FG)' '#Polyarthropathies'
 '#Spondylopathies' '3mm asymmetry angle (left)'
 '3mm asymmetry angle (right)' '3mm asymmetry index (left)'
 '3mm asymmetry index (right)' '3mm cylindrical power (left)'
 '3mm cylindrical power (right)' '3mm cylindrical power angle (left)'
 '3mm cylindrical power angle (right)'
 '3mm index of best keratometry results (left)'
 '3mm index of best keratometry results (right)'
 '3mm regularity index (left)' '3mm regularity index (right)'
 '3mm strong meridian (left)' '3mm strong meridian (right)'
 '3mm strong meridian angle (left)' '3mm strong meridian angle (right)'
 '3mm weak meridian (left)' '3mm weak meridian (right)'
 '3mm weak meridian angle (left)' '3mm weak meridian angle (right)'
 '6mm asymmetry angle (left)' '6mm asymmetry angle (right)'
 '6mm asymmetry index (left)' '6mm asymmetry index (right)'
 '6mm cylindrical po

In [15]:
ts = ts_df.groupby(['Reference SNP cluster ID', 'Trait']).mean()
ts = ts.reset_index()

In [16]:
print(len(trait_values))
print(len(SNP_values))

3815
2090230


In [17]:
# ts[(ts['Trait']=='3mm strong meridian (right)') & (ts['Reference SNP cluster ID']=='rs570860110')]

In [18]:
# ts[(ts['Trait']=='3mm strong meridian (right)') & (ts['Reference SNP cluster ID']=='rs570860110')]

In [19]:
ts['trait_ind'] = ts.groupby(['Trait']).ngroup()
ts['snp_ind'] = ts.groupby(['Reference SNP cluster ID']).ngroup()

In [20]:
ts.head(20)

,Reference SNP cluster ID,Trait,pval_mlog,trait_ind,snp_ind
0,10:100014847_CT_C,Impedance of arm (left),0.038691,1436,0
1,10:100014847_CT_C,Impedance of arm (right),0.029557,1437,0
2,10:100014847_CT_C,Impedance of leg (left),0.027193,1438,0
3,10:100014847_CT_C,Impedance of leg (right),0.031024,1439,0
4,10:100014847_CT_C,Impedance of whole body,0.035546,1440,0
5,10:100175766_CT_C,Job SOC coding: Leisure and travel service occ...,0.023613,1590,1
6,10:100394295_CTCTTT_C,Diagnoses - main ICD10: G12 Spinal muscular at...,0.025467,517,2
7,10:100394295_CTCTTT_C,Job SOC coding: Chiropodists,0.023281,1501,2
8,10:100394295_CTCTTT_C,"Job coding: chiropodist, podiatrist",0.023747,1798,2
9,10:100429709_CA_C,Treatment/medication code: brinzolamide,0.029572,3079,3


In [21]:
columns = ts['snp_ind'].to_list()
rows = ts['trait_ind'].to_list()
pval_mlog_data = ts['pval_mlog'].to_list()

In [22]:
data_coo = sp.sparse.coo_matrix((pval_mlog_data, (rows, columns)), shape=(len(trait_values), len(SNP_values)))

In [201]:
print(data_coo.max())

1.0


In [202]:
#splitting data into train and test
(train, test) = cross_validation.random_train_test_split(data_coo, test_percentage=0.2, random_state=None)

In [203]:
data = {'train': train, 'test': test, 'SNP_values' : SNP_values}

In [204]:
#creating model
gwas_model_e30_d10 = LightFM(no_components=10, loss='warp')

In [206]:
#training model
gwas_model_e30_d10.fit(data['train'], epochs=30, num_threads=2)

In [207]:
pickle_out_data = open('/home/ctrader/Hold/GWAS/gwas_data_e30_d10_CT20211004.pickle', 'wb')
pickle.dump(data, pickle_out_data)
pickle_out_data.close()

pickle_out_model = open('/home/ctrader/Hold/GWAS/gwas_model_e30_d10_CT20211004.pickle', 'wb')
pickle.dump(gwas_model_e30_d10, pickle_out_model)
pickle_out_model.close()

In [39]:
pickle_in = open('/home/ctrader/Hold/GWAS/gwas_model_e30_d10_CT20211004.pickle', 'rb')
gwas_model_e30_d10=pickle.load(pickle_in)

pickle_in = open('/home/ctrader/Hold/GWAS/gwas_data_e30_d10_CT20211004.pickle', 'rb')
data=pickle.load(pickle_in)

In [55]:
full_data = data_coo
trait = 'Impedance of leg (left)'
ts[ts['Trait']==trait]
trait_ind = ts[ts['Trait'] == trait]['trait_ind'].tolist()[0]

known_ranked = list(np.squeeze(np.asarray(np.argsort(-data_coo.todense()[trait_ind, :]))))
# print(known_ranked[0])
n_traits, n_snps = data['train'].shape
scores = gwas_model.predict(trait_ind, np.arange(n_snps))
top_SNPs = np.argsort(-scores)
# print(top_SNPs[2090232])



1431229
1049063


In [169]:
#can delete
# chrom = df[df['Reference SNP cluster ID']=='rs559230165']['Chromosome'].to_list()[0]
# p1 = df[df['Reference SNP cluster ID']=='rs559230165']['p1'].to_list()[0]
# p2 = df[df['Reference SNP cluster ID']=='rs559230165']['p2'].to_list()[0]
# print(p1)
df[(df['Reference SNP cluster ID']=='rs199639620') & (df['Trait']=='#Deforming dorsopathies')]

,Chromosome,p1,p2,Reference SNP cluster ID,p-value,Trait,pval_mlog
6416519,chr19,49489939,49489940,rs199639620,0.00000e+00,#Deforming dorsopathies,1.0


In [5]:
#number of diseases and genes in training dataset
n_traits, n_snps = data['train'].shape
print(n_traits, n_snps)

3815 2090230


In [29]:
#function to recommend SNPs for a given trait. Takes in the model, data, and a list of
#traits for SNPs to be recommend for. For each trait, prints out known associated SNPs
#and a list of recommended genes. Function modeled after tutorial 3, 'Recommendation Systems
#- Learn Python for Data Science' on LightFM github. 

def sample_recommendation(model, data, traits, recommend_num=5, full_data=data_coo):
    
    
    #generate gene recommendations for each disease
    for trait in traits:
        trait_ind = ts[ts['Trait'] == trait]['trait_ind'].tolist()[0]
        #genes known to relate to the disease
        known_ranked = list(np.squeeze(np.asarray(np.argsort(-data_coo.todense()[trait_ind, :]))))
#         print(known_ranked)
        
        #genes model predicts that relate to disease
        scores = model.predict(trait_ind, np.arange(n_snps))
        #rank from most related to least
        top_SNPs = np.argsort(-scores)
        
        #print results
        print('Disease: ' + trait + ' (' + str(trait_ind) +')')
        print('   Known Positives:')
        
#         known_pos = [] 
#         known_pval_mlog = []
        ts_dense = full_data.todense()
        known_pos = ts[(ts['Trait']==trait)]['Reference SNP cluster ID'].to_list()
        for snp in known_ranked[:5]:
            pval_mlog = ts_dense[trait_ind, snp]*max_val
            if pval_mlog == 0:
                break
            s = ts[ts['snp_ind'] == snp]['Reference SNP cluster ID'].tolist()[0]
            chrom = df[df['Reference SNP cluster ID']==s]['Chromosome'].to_list()[0]
            p1 = df[df['Reference SNP cluster ID']==s]['p1'].to_list()[0]
            p2 = df[df['Reference SNP cluster ID']==s]['p2'].to_list()[0]
            pval = df[(df['Reference SNP cluster ID']==s) & (df['Trait']==trait)]['p-value'].to_list()[0]
            print('      Reference SNP cluster ID: %s, %s:%s-%s, p-value = %s' %(s, chrom, p1, p2, pval))
        print('      ...')
        
        print()
        print('   Recommended:')
        count = 0
        for snp in top_SNPs[:]:
            s = ts[ts['snp_ind'] == snp]['Reference SNP cluster ID'].tolist()[0]
            if s not in known_pos:
                chrom = df[df['Reference SNP cluster ID']==s]['Chromosome'].to_list()[0]
                p1 = df[df['Reference SNP cluster ID']==s]['p1'].to_list()[0]
                p2 = df[df['Reference SNP cluster ID']==s]['p2'].to_list()[0]
                print('      Reference SNP cluster ID: %s, %s:%s-%s' %(s, chrom, p1, p2))
                count += 1
            if count == recommend_num:
                break
                
        print()
            
sample_recommendation(gwas_model_e30_d10, data, ['#Arthrosis'])

Disease: #Arthrosis (0)
   Known Positives:
      Reference SNP cluster ID: rs559230165, chr1:219580670-219580671, p-value = 2.51849e-11
      Reference SNP cluster ID: rs1415293, chr1:219556664-219556665, p-value = 3.90813e-11
      Reference SNP cluster ID: rs4846303, chr1:219557457-219557458, p-value = 3.98795e-11
      Reference SNP cluster ID: rs2785998, chr1:219561656-219561657, p-value = 4.22478e-11
      Reference SNP cluster ID: rs1337101, chr1:219552758-219552759, p-value = 4.29033e-11
      ...

   Recommended:
      Reference SNP cluster ID: rs7182563, chr15:83879994-83879995
      Reference SNP cluster ID: rs6142376, chr20:35404958-35404959
      Reference SNP cluster ID: rs979543, chr15:83683812-83683813
      Reference SNP cluster ID: rs1865689, chr2:24885328-24885329
      Reference SNP cluster ID: rs13003430, chr2:24911475-24911476



In [127]:
ts[(ts['trait_ind']==1438)&(ts['Reference SNP cluster ID']=='rs78277448')]

,Reference SNP cluster ID,Trait,pval_mlog,trait_ind,snp_ind


In [ ]:
#AUC score
auc_score = auc_score(gwas_model_e30_d10, data['test'])

In [ ]:
n_traits, n_snp = data['train'].shape
scores = model.predict(15, np.arange(n_SNPs))
top_genes = data['gene_ids'][np.argsort(-scores)]

In [19]:
df[(df['Reference SNP cluster ID']=='rs570860110')&(df['Trait']=='3mm strong meridian (right)')]

,Chromosome,p1,p2,Reference SNP cluster ID,p-value,Trait,pval_mlog
1009286,chr1,202817700,202817701,rs570860110,1.63190e-11,3mm strong meridian (right),0.033397
1009292,chr1,202817700,202817701,rs570860110,3.26722e-11,3mm strong meridian (right),0.032463
1009300,chr1,202817700,202817701,rs570860110,4.72082e-12,3mm strong meridian (right),0.035064
1009308,chr1,202817700,202817701,rs570860110,9.41552e-12,3mm strong meridian (right),0.034136


In [42]:
test_dense = data['test'].todense()
train_dense = data['train'].todense()

In [41]:
#Calculating precision and recall for test set
test_precision_list_top = []
test_recall_list_top = []
for k in [1, 10, 100, 1000, 10000]: #
    print('k =', k)
    num = []
    n_known_pos = []
    for trait in range(n_traits): 
        if trait % 500 == 0:
            print(trait, '/', n_traits)
        scores = gwas_model_e30_d10.predict(trait, np.arange(n_snps)) 
        top_snps = [np.argsort(-scores)][0][0:k]
        known_pos = [i for i, snp_pos in enumerate(np.squeeze(np.asarray(test_dense[trait]))) if snp_pos > 0]
        top_in_known = len([i for i in known_pos if i in top_snps])
        num.append(top_in_known)
        known_pos_len = len(known_pos)
        if (known_pos_len == 0) and (top_in_known == 0):
            known_pos_len = 1
        n_known_pos.append(known_pos_len)
            
    num = np.array(num)
    precision = (num/k).mean()
    recall = (num/n_known_pos).mean()
    print('Precision for k=%s : %s' %(k, precision))
    print('Recall for k=%s : %s' %(k, recall))
    test_precision_list_top.append(precision)
    test_recall_list_top.append(recall)

k = 1
0 / 3815
500 / 3815
1000 / 3815
1500 / 3815
2000 / 3815
2500 / 3815
3000 / 3815
3500 / 3815
Precision for k=1 : 0.03669724770642202
Recall for k=1 : 0.00020642155121893197
k = 10
0 / 3815
500 / 3815
1000 / 3815
1500 / 3815
2000 / 3815
2500 / 3815
3000 / 3815
3500 / 3815
Precision for k=10 : 0.0272608125819135
Recall for k=10 : 0.0015304969110941317
k = 100
0 / 3815
500 / 3815
1000 / 3815
1500 / 3815
2000 / 3815
2500 / 3815
3000 / 3815
3500 / 3815
Precision for k=100 : 0.0178348623853211
Recall for k=100 : 0.009815519200025173
k = 1000
0 / 3815
500 / 3815
1000 / 3815
1500 / 3815
2000 / 3815
2500 / 3815
3000 / 3815
3500 / 3815
Precision for k=1000 : 0.013075753604193972
Recall for k=1000 : 0.044267974585764494
k = 10000
0 / 3815
500 / 3815
1000 / 3815
1500 / 3815
2000 / 3815
2500 / 3815
3000 / 3815
3500 / 3815
Precision for k=10000 : 0.009081572739187417
Recall for k=10000 : 0.19320361731253163


In [ ]:
#Calculating precision and recall for train set
train_precision_list_top = []
train_recall_list_top = []
for k in [1, 10, 100, 1000, 10000]: #
    print('k =', k)
    num = []
    n_known_pos = []
    for trait in range(n_traits): 
        if trait % 500 == 0:
            print(trait, '/', n_traits)
        scores = gwas_model_e30_d10.predict(trait, np.arange(n_snps)) 
        top_snps = [np.argsort(-scores)][0][0:k]
        known_pos = [i for i, snp_pos in enumerate(np.squeeze(np.asarray(train_dense[trait]))) if snp_pos > 0]
        top_in_known = len([i for i in known_pos if i in top_snps])
        num.append(top_in_known)
        known_pos_len = len(known_pos)
        if (known_pos_len == 0) and (top_in_known == 0):
            known_pos_len = 1
        n_known_pos.append(known_pos_len)
            
    num = np.array(num)
    precision = (num/k).mean()
    recall = (num/n_known_pos).mean()
    print('Precision for k=%s : %s' %(k, precision))
    print('Recall for k=%s : %s' %(k, recall))
    train_precision_list_top.append(precision)
    train_recall_list_top.append(recall)

k = 1
0 / 3815
500 / 3815
1000 / 3815
1500 / 3815
2000 / 3815
2500 / 3815
3000 / 3815


In [99]:
# counter = 0
# for trait in trait_values:
# #     print(trait)
#     trait_df = ts_df[ts_df['Trait']==trait].set_index('Reference SNP cluster ID')
#     duplicate_list = np.unique(list(trait_df[trait_df.index.duplicated()].index))
#     if len(duplicate_list) > 0:
#         entry_list = np.array([])
#         for duplicate in duplicate_list:
# #             print(duplicate)
#             ave = np.mean(trait_df.loc[duplicate]['pval_mlog'])
#             trait_df = trait_df.drop(duplicate)
#             np.append(entry_list, [trait, ave, duplicate], axis=0)
#         entry = pd.DataFrame(list(entry_list), columns=['Trait', 'pval_mlog', 'Reference SNP cluster ID'])
#         entry = entry.set_index('Reference SNP cluster ID')
# #         print('entry done')
#         trait_df = trait_df.append(entry)
# #         print('df done')
#     ts_matrix.loc[trait] = trait_df['pval_mlog']
# #     print('main df done')
#     if counter%50 == 0:
#         print('Trait %s done' %(counter))
#     counter+=1

# ts_np = ts_matrix.to_numpy()
# ts_np = np.array(ts_np, dtype=float)
# ts_np[np.isnan(ts_np)] = 0
# ts_coo = sp.sparse.coo_matrix(ts_np)
# print(ts_coo)

In [27]:
ts[(ts['Reference SNP cluster ID']=='rs4502182')]

,Reference SNP cluster ID,Trait,pval_mlog,trait_ind,snp_ind
6711701,rs4502182,Body fat percentage,0.030429,201,1188245
6711702,rs4502182,Comparative height size at age 10,0.083777,302,1188245
6711703,rs4502182,"Forced expiratory volume in 1-second (FEV1), p...",0.037613,1196,1188245
6711704,rs4502182,Impedance of arm (left),0.053102,1436,1188245
6711705,rs4502182,Impedance of arm (right),0.045766,1437,1188245
6711706,rs4502182,Impedance of leg (left),0.064415,1438,1188245
6711707,rs4502182,Impedance of leg (right),0.069979,1439,1188245
6711708,rs4502182,Impedance of whole body,0.068557,1440,1188245
6711709,rs4502182,Sitting height,0.033438,2896,1188245
6711710,rs4502182,Standing height,0.114045,2941,1188245


In [35]:
# new model
ts[(ts['Reference SNP cluster ID']=='rs13003430')]

,Reference SNP cluster ID,Trait,pval_mlog,trait_ind,snp_ind
2216112,rs13003430,Age when periods started (menarche),0.022840,93,400353
2216113,rs13003430,Arm fat mass (left),0.090019,125,400353
2216114,rs13003430,Arm fat mass (right),0.090886,126,400353
2216115,rs13003430,Arm fat percentage (left),0.148816,127,400353
2216116,rs13003430,Arm fat percentage (right),0.143625,128,400353
2216117,rs13003430,Body fat percentage,0.115039,201,400353
2216118,rs13003430,Body mass index (BMI),0.132205,202,400353
2216119,rs13003430,"Cereal type: Bran cereal (e.g. All Bran, Branf...",0.022756,267,400353
2216120,rs13003430,Comparative body size at age 10,0.327479,301,400353
2216121,rs13003430,Comparative height size at age 10,0.043401,302,400353
